In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# การลดความผิดพลาดด้วย Tensor Network (TEM): Qiskit Function โดย Algorithmiq
*ดู[เอกสารอ้างอิง API](https://docs.quantum.ibm.com/api/functions/algorithmiq-tem)*

> **Note:** Qiskit Functions เป็นฟีเจอร์ทดลองที่ใช้ได้เฉพาะผู้ใช้แผน IBM Quantum&reg; Premium Plan, Flex Plan และ On-Prem (ผ่าน IBM Quantum Platform API) Plan เท่านั้น อยู่ในสถานะ preview release และอาจมีการเปลี่ยนแปลงได้


<Accordion>
<AccordionItem title="เวอร์ชันของแพ็กเกจ">

โค้ดในหน้านี้พัฒนาโดยใช้ requirements ต่อไปนี้
แนะนำให้ใช้เวอร์ชันเหล่านี้หรือใหม่กว่า

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
## ภาพรวม
วิธี Tensor-network Error Mitigation (TEM) ของ Algorithmiq เป็นอัลกอริทึมแบบ hybrid
quantum-classical ที่ออกแบบมาเพื่อทำการลดความผิดพลาดของ noise ทั้งหมดในขั้นตอน
post-processing แบบ classical ด้วย TEM ผู้ใช้สามารถคำนวณค่าความคาดหวัง
(expectation values) ของ observable ได้โดยลดข้อผิดพลาดที่เกิดจาก noise
อันหลีกเลี่ยงไม่ได้บนฮาร์ดแวร์ควอนตัม ด้วยความแม่นยำและประสิทธิภาพด้านต้นทุนที่เพิ่มขึ้น
ทำให้เป็นตัวเลือกที่น่าสนใจอย่างมากสำหรับนักวิจัยด้านควอนตัมและผู้ปฏิบัติงานในอุตสาหกรรม

วิธีการนี้ประกอบด้วยการสร้าง tensor network ที่แทนค่าผกผันของช่องทาง noise ส่วนกลาง
ที่กระทบต่อสถานะของโปรเซสเซอร์ควอนตัม แล้วนำ map นั้นไปใช้กับผลการวัดที่ครอบคลุมข้อมูล
อย่างสมบูรณ์ (informationally complete) ที่เก็บจากสถานะที่มี noise เพื่อให้ได้ตัวประมาณค่า
แบบไม่เอนเอียง (unbiased estimators) สำหรับ observable

ในฐานะข้อได้เปรียบ TEM ใช้ประโยชน์จากการวัดที่ครอบคลุมข้อมูลอย่างสมบูรณ์เพื่อเข้าถึง
observable ที่ผ่านการลด noise จำนวนมาก และมี sampling overhead บนฮาร์ดแวร์ควอนตัม
ที่เหมาะสมที่สุด ตามที่อธิบายใน Filippov et al. (2023),
[arXiv:2307.11740](https://arxiv.org/abs/2307.11740) และ Filippov et al. (2024),
[arXiv:2403.13542](https://arxiv.org/abs/2403.13542) overhead ของการวัดหมายถึงจำนวน
การวัดเพิ่มเติมที่ต้องการเพื่อทำการลด noise อย่างมีประสิทธิภาพ ซึ่งเป็นปัจจัยสำคัญ
ในความเป็นไปได้ของการคำนวณควอนตัม ดังนั้น TEM จึงมีศักยภาพในการเปิดใช้งาน
quantum advantage ในสถานการณ์ที่ซับซ้อน เช่น การประยุกต์ใช้ในสาขา quantum chaos,
many-body physics, Hubbard dynamics และการจำลองเคมีของโมเลกุลขนาดเล็ก

คุณสมบัติหลักและประโยชน์ของ TEM สรุปได้ดังนี้:

1.  **Measurement overhead ที่เหมาะสมที่สุด**: TEM เหมาะสมที่สุดเมื่อเทียบกับ
[ขอบเขตทางทฤษฎี](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601)
หมายความว่าไม่มีวิธีใดที่จะมี measurement overhead น้อยกว่านี้ได้ กล่าวอีกนัยหนึ่ง
TEM ต้องการจำนวนการวัดเพิ่มเติมน้อยที่สุดในการลด noise ซึ่งหมายความว่า TEM ใช้
quantum runtime น้อยที่สุด
2.  **ความคุ้มค่าด้านต้นทุน**: เนื่องจาก TEM จัดการการลด noise ทั้งหมดในขั้นตอน
post-processing จึงไม่จำเป็นต้องเพิ่ม circuit เพิ่มเติมบนคอมพิวเตอร์ควอนตัม
ซึ่งไม่เพียงทำให้การคำนวณถูกลงเท่านั้น แต่ยังลดความเสี่ยงในการเพิ่มข้อผิดพลาด
จากความไม่สมบูรณ์ของอุปกรณ์ควอนตัมด้วย
3.  **การประมาณค่า observable หลายตัว**: ด้วยการวัดที่ครอบคลุมข้อมูลอย่างสมบูรณ์
TEM ประมาณค่า observable หลายตัวได้อย่างมีประสิทธิภาพจากข้อมูลการวัดชุดเดียวกัน
จากคอมพิวเตอร์ควอนตัม
4.  **การลด readout error**: TEM Qiskit Function ยังมี
[วิธีการลด readout error แบบเฉพาะ](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154)
ที่สามารถลด readout error ได้อย่างมีนัยสำคัญหลังการ calibrate ในเวลาสั้น
5.  **ความแม่นยำ**: TEM ปรับปรุงความแม่นยำและความน่าเชื่อถือของการจำลองควอนตัม
ดิจิทัลอย่างมีนัยสำคัญ ทำให้อัลกอริทึมควอนตัมมีความแม่นยำและน่าเชื่อถือมากขึ้น
## คำอธิบาย
ฟังก์ชัน TEM ช่วยให้ได้ค่าความคาดหวังที่ผ่านการลด noise สำหรับ observable หลายตัว
บน quantum Circuit ด้วย sampling overhead น้อยที่สุด Circuit จะถูกวัดด้วย
informationally complete positive operator-valued measure (IC-POVM) และผลการวัด
ที่เก็บได้จะถูกประมวลผลบนคอมพิวเตอร์ classical การวัดนี้ใช้ในการดำเนินการวิธี
tensor network และสร้าง noise-inversion map ฟังก์ชันนำ map ที่กลับค่า noise ของ
Circuit ทั้งหมดโดยใช้ tensor network เพื่อแทนชั้นที่มี noise

![TEM schematics](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "การประมาณ observable O ที่ผ่านการลด noise ผ่านการประมวลผลผลการวัดของโปรเซสเซอร์ควอนตัมที่มี noise U และ N แทนการดำเนินการควอนตัมในอุดมคติและ noise map ที่เกี่ยวข้อง ซึ่งโดยทั่วไปอาจไม่ใช่แบบ local (และขยายเป็นกล่องสีเทา) D แทน tensor ของตัวดำเนินการที่เป็น dual กับ effect ในการวัด IC โมดูลลด noise M คือ tensor network ที่ถูก contract อย่างมีประสิทธิภาพจากกลางออกไปสองด้าน การ contraction ครั้งแรกแสดงด้วยเส้นสีม่วงแบบจุด ครั้งที่สองด้วยเส้นประ และครั้งที่สามด้วยเส้นทึบ")

เมื่อ Circuit ถูกส่งไปยังฟังก์ชัน จะถูก transpile และ optimize เพื่อลดจำนวนชั้นที่มี
two-qubit gate (gate ที่มี noise มากที่สุดบนอุปกรณ์ควอนตัม) Noise ที่กระทบต่อชั้นต่างๆ
จะถูกเรียนรู้ผ่าน
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
โดยใช้โมเดล noise แบบ sparse Pauli-Lindblad ตามที่อธิบายใน E. van den Berg, Z.
Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866)

โมเดล noise เป็นคำอธิบายที่แม่นยำของ noise บนอุปกรณ์ที่สามารถจับคุณสมบัติ
ที่ละเอียดอ่อนได้ รวมถึง qubit cross-talk อย่างไรก็ตาม noise บนอุปกรณ์อาจผันผวน
และเปลี่ยนแปลง และ noise ที่เรียนรู้ไว้อาจไม่แม่นยำ ณ เวลาที่ทำการประมาณค่า
ซึ่งอาจส่งผลให้ได้ผลลัพธ์ที่ไม่แม่นยำ
## เริ่มต้นใช้งาน
ยืนยันตัวตนด้วย [IBM Quantum Platform API key](http://quantum.cloud.ibm.com/) และเลือกฟังก์ชัน TEM ดังนี้ (ตัวอย่างนี้สมมติว่า[บันทึกบัญชีของคุณ](/guides/functions#install-qiskit-functions-catalog-client)ไว้ในสภาพแวดล้อม local แล้ว)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")
# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [ ]:
# Load your function
tem = catalog.load(tem_function_name)

ใช้ Qiskit Serverless APIs เพื่อตรวจสอบสถานะของ Qiskit Function workload:

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device
# reported by IBM if not specified
backend_name = "ibm_marrakesh"

# Run the TEM function (uses around three minutes of QPU time)
job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Use the Qiskit Serverless APIs to check your Qiskit Function workload's status:

In [3]:
# Print the ID so you can use it later, if necessary
print(job.job_id)
print(job.status())

30db31ed-d252-4ca8-a5f0-5eb9b5075a4c
QUEUED


ดึงผลลัพธ์ได้ดังนี้:

In [4]:
result = job.result()
evs = result[0].data.evs
print(evs[0])

0.09417719217134088


<Admonition type="info">
  The expected value for the noiseless circuit for the given operator should be around `0.18409094298943401`.
</Admonition>

## Get support

Reach out to [qiskit_ibm@algorithmiq.fi](mailto:qiskit_ibm@algorithmiq.fi)

Be sure to include the following information:
- Qiskit Function Job ID (`qiskit-ibm-catalog`), `job.job_id`
- A detailed description of the issue
- Any relevant error messages or codes
- Steps to reproduce the issue

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Algorithmiq Tensor-network error mitigation](https://quantum.ibm.com/functions?id=4b1b9d76-c18b-4788-b70b-15125111fbe6).
- Visit the [API reference](/docs/api/functions/algorithmiq-tem) for this Qiskit Function.

</Admonition>